# SingleTaskVariationalGP: scalable approximate GP

Exact GPの学習が重くなるデータ量に対して、`SingleTaskVariationalGP`をinducing-point variational inferenceで学習します。

## このNotebookで確認すること

- inducing pointsを持つvariational surrogateの構築
- `VariationalELBO`と`num_data`の意味
- minibatch学習の実際のstate transition
- approximate posteriorと不確実性の確認
- Exact GPとの使い分け

関連理論:

- [Variational GP](../../docs/theory/09_variational_gp.md)
- [Model selection](../../docs/theory/13_model_selection.md)

このNotebookは**Tier B: Model Family**です。目的はvariational inference contractを正しく示すことで、`01_single_task_gp.ipynb`のBO loopを複製することではありません。学習後のmodelはBoTorch-compatible posteriorを返すため、互換なacquisitionへ接続できます。完全な基本BO loopは[01_single_task_gp.ipynb](01_single_task_gp.ipynb)を参照してください。

## 1. このモデルを使う場面

Exact GP の計算量が実用上のボトルネックになる場合に variational GP を検討します。近似精度と計算量は主に inducing points の数と配置に依存します。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from torch.utils.data import DataLoader, TensorDataset

from robotorchan.models import SingleTaskVariationalGP

torch.manual_seed(0)
dtype = torch.double

### Exact GPと変分GPの違い

Exact GPは観測数 $N$ に応じて共分散行列を扱い、標準的な密行列計算では学習コストが概ね $O(N^3)$ になります。変分GPは $M$ 個（通常 $M\ll N$）の**inducing points** を使って潜在関数の事後分布を近似し、大規模データへの適用を目指します。

変分近似では、学習が速くなる可能性がある一方、inducing pointsの数・配置、学習回数、最適化設定によって予測精度や不確実性の校正が変わります。常にExact GPより優れるわけではありません。

## 2. 合成学習データ

In [ ]:
def objective(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.0 * torch.pi * x) + 0.25 * torch.cos(6.0 * torch.pi * x)

n_train = 400
train_X = torch.rand(n_train, 1, dtype=dtype)
train_Y = objective(train_X) + 0.08 * torch.randn(n_train, 1, dtype=dtype)

train_X.shape, train_Y.shape

### この合成データで何を学ぶか

真の関数 $f(x)=\sin(2\pi x)+0.25\cos(6\pi x)$ に観測ノイズを加えた400点を作ります。最適化したいのはノイズを含む観測値そのものではなく、未知の関数の高い値を与える設計点です。

学習用の `train_Y` にはノイズが含まれますが、後半の `objective(candidate)` はノイズなしで評価する簡略化例です。

## 3. モデル構築

ここでは 400 点の学習データに対して 48 個の inducing points を使用します。

In [ ]:
model = SingleTaskVariationalGP(
    train_X=train_X,
    train_Y=train_Y,
    inducing_points=48,
)
model

### Inducing pointsの役割

`inducing_points=48` は、400観測に対して48個の代表的な潜在点を用いて事後分布を近似する設定です。これは観測を48点に間引く操作ではありません。すべての学習データをELBOの学習に利用しながら、近似の表現力を制御します。

実務ではinducing pointsを増やすと近似表現力が向上する場合がありますが、メモリ・計算量も増えます。

## 4. robotorchan 共通 API

wrapper はコンストラクタに渡したテンソルを保持し、ELBO を生成する `make_mll()` を提供します。上流の variational constructor には `train_Yvar` がないため、`raw_train_Yvar` は `None` です。

In [ ]:
print("raw_train_X:", model.raw_train_X.shape)
print("raw_train_Y:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)
print("raw_data_names:", model.raw_data_names)
print("supports_mll:", model.supports_mll)

mll = model.make_mll(num_data=n_train)
type(mll).__name__

### ELBOと`num_data`の意味

変分推論では、近似事後分布 $q(u)$ と事前分布 $p(u)$ のずれを抑えつつ、観測を説明するように**Evidence Lower Bound（ELBO）**を最大化します。概念的には

$$\mathrm{ELBO}=\mathbb E_q[\log p(y\mid f)]-\mathrm{KL}(q(u)\Vert p(u))$$

と表せます。`num_data=n_train` は全学習データ数を示し、minibatchの尤度項を全データに対応するようにスケーリングするために必要です。**`batch_size=64` を `num_data` に指定するわけではありません。**

## 5. Minibatch による variational 学習

`fit()` は保存した観測データを使い、`num_data` を全件数に保ったELBOをAdamで最適化します。`batch_size` を指定するとminibatch学習になります。


In [ ]:
loader = DataLoader(
    TensorDataset(train_X, train_Y),
    batch_size=64,
    shuffle=True,
)

model.fit(epochs=60, lr=0.03, batch_size=64)
print("Variational GP trained with minibatch ELBO.")


In [ ]:
# The fit() method performs the ELBO optimization internally.
print("Training completed; posterior is available below.")

### 学習の流れとminibatch

`model.fit(epochs=60, lr=0.03, batch_size=64)` が内部でミニバッチを取り出し、ELBOに基づいてモデルのパラメータを更新します。Notebook中で `DataLoader` を作っていますが、**この `loader` を `fit()` に渡しているわけではありません**。学習の実行主体は `fit()` です。

学習が不安定な場合は、まず学習率、epoch数、inducing points数を見直します。

## 6. Posterior 予測

In [ ]:
model.eval()
model.likelihood.eval()

test_X = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.sqrt().squeeze(-1)

lower = mean - 1.96 * std
upper = mean + 1.96 * std

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), s=8, alpha=0.25, label="observations")
plt.plot(test_X.squeeze(-1), objective(test_X).squeeze(-1), linestyle="--", label="true function")
plt.plot(test_X.squeeze(-1), mean, label="posterior mean")
plt.fill_between(test_X.squeeze(-1), lower, upper, alpha=0.2, label="95% interval")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("SingleTaskVariationalGP posterior")
plt.show()

### 変分事後分布の読み方

`posterior.mean` はモデルが予測する潜在関数の平均、`posterior.variance` はその不確実性です。帯の $\mu(x)\pm1.96\sigma(x)$ は**近似事後分布の潜在関数に関する約95%区間**であり、観測ノイズを含む将来の測定値の区間ではありません。

変分近似の誤差やモデルの仮定によって、区間の実際の被覆率が95%になる保証はありません。

## 7. Exact GP との使い分け

小規模データでは exact inference が単純で統計的にも効率的な `SingleTaskGP` が通常は第一候補です。データ量が増えて covariance の分解が重くなると `SingleTaskVariationalGP` が有力になります。inducing points の数は計算速度と近似精度のトレードオフを制御します。

## 8. Bayesian Optimizationへの接続

学習済み`SingleTaskVariationalGP`はposteriorを通じてBoTorch acquisitionへ接続できます。一方、variational modelではExact GPと学習手順・近似誤差・更新コストが異なります。

このNotebookではmodel family固有のminibatch variational trainingを主題とし、候補生成のためだけに`01_single_task_gp.ipynb`と同じqLogEI loopを複製しません。後続のworkflow Notebookでは、surrogateを差し替えたときもModel → Posterior → Acquisition → Candidateというcross-layer contractが保たれることを確認します。

## 9. Limitations and caveats

inducing pointsの数・配置とoptimizer設定は近似品質と計算時間に影響します。この400点・48 inducing points・60 epochという設定はAPIと学習フローを確認するための小さな例であり、一般的な推奨値ではありません。

### EIで次の評価点を選ぶ

学習した変分GPをBoTorch互換の獲得関数 `qLogExpectedImprovement` に渡します。EIは、現時点の最良観測値 `best_f` を超える改善の期待値に基づいて、次に評価する候補を選びます。

`q=1` は候補を1点提案する指定です。変分GPでも **モデル → 事後分布 → 獲得関数 → 候補提案 → 評価** の流れは同じです。

## Acquisition, optimization, and candidate evaluation

変分GPのposteriorからqLogEIを構築し、探索区間内の候補を提案・評価します。学習は上記の既存ELBO minibatch loopを維持します。

In [ ]:
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
acquisition = qLogExpectedImprovement(model=model, best_f=train_Y.max())
candidate, acquisition_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
candidate_Y = objective(candidate)
assert candidate.shape == (1, 1)
assert torch.all((candidate >= bounds[0]) & (candidate <= bounds[1]))
print("candidate:", candidate)
print("acquisition value:", acquisition_value)
print("evaluated objective:", candidate_Y)


### Mixed変分GPの注意点

Mixed版では連続変数 $x$ とカテゴリ $c\in\{0,1\}$ を同時に扱います。`cat_dims=[1]` は2列目がカテゴリであることを示します。カテゴリは連続値として補間せず、`fixed_features_list` で有効なカテゴリを列挙して候補を探索します。

後半の例ではカテゴリ効果として $0.2c$ を加えています。これは**設計変数としてのカテゴリ**であり、Notebook 04のtask IDとは異なります。

## 10. Mixed variational GPの学習と候補提案

カテゴリ特徴量はone-hotへ置換せず、`cat_dims`を指定したnative mixed covarianceで学習します。単一タスクと同様にminibatch ELBOを使い、カテゴリ値を固定した獲得関数最適化で候補を提案します。短い学習設定はAPI実行確認用です。


In [ ]:
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import MixedSingleTaskVariationalGP

mixed_train_X = torch.cat(
    [train_X, (torch.arange(n_train) % 2).to(train_X).unsqueeze(-1)],
    dim=-1,
)
mixed_train_Y = objective(train_X) + 0.2 * mixed_train_X[:, 1:2]
mixed_model = MixedSingleTaskVariationalGP(
    train_X=mixed_train_X,
    train_Y=mixed_train_Y,
    cat_dims=[1],
    inducing_points=48,
)
mixed_loader = DataLoader(
    TensorDataset(mixed_train_X, mixed_train_Y),
    batch_size=64,
    shuffle=True,
)
mixed_model.fit(epochs=30, lr=0.03, batch_size=64)

mixed_model.eval()
mixed_model.likelihood.eval()
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
mixed_acquisition = qLogExpectedImprovement(
    model=mixed_model,
    best_f=mixed_train_Y.max(),
)
mixed_candidate, mixed_acquisition_value = optimize_acqf_mixed(
    acq_function=mixed_acquisition,
    bounds=mixed_bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
    fixed_features_list=[{1: 0.0}, {1: 1.0}],
)
assert mixed_candidate.shape == (1, 2)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
mixed_candidate_Y = (
    objective(mixed_candidate[:, :1]) + 0.2 * mixed_candidate[:, 1:2]
)
print("Mixed variational candidate:", mixed_candidate)
print("Acquisition value:", mixed_acquisition_value)
print("Objective:", mixed_candidate_Y)


## 11. 追加実装例：候補の観測を追加して再学習する

1回の候補提案で終わらず、実際のBOでは新しい観測をデータに追加し、サロゲートを更新します。ここでは元の学習データを変更せず、`candidate` と `candidate_Y` を加えた**新しい変分GP**を作ります。

新しいモデルを構築するため、これはwarm-startではなく**再初期化して再学習する例**です。既存の `fit()` APIを再利用し、複雑な学習ループを追加しません。

In [ ]:
updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y.detach()], dim=0)

updated_model = SingleTaskVariationalGP(
    train_X=updated_X,
    train_Y=updated_Y,
    inducing_points=48,
)
updated_model.fit(epochs=20, lr=0.03, batch_size=64)
updated_model.eval()
updated_model.likelihood.eval()

with torch.no_grad():
    updated_posterior = updated_model.posterior(candidate.detach())
assert updated_X.shape[0] == n_train + 1
assert updated_Y.shape[0] == n_train + 1
print("new observation:", candidate_Y)
print("updated posterior mean at candidate:", updated_posterior.mean)
print("updated posterior std at candidate:", updated_posterior.variance.clamp_min(0).sqrt())